# ReXKG Pipeline (PyHealth Style)

This notebook shows a PyHealth-native ReXKG workflow similar to other examples:

1. Load dataset
2. Set tasks
3. Build sample datasets
4. Create dataloaders
5. Run model forward smoke test

It also includes optional RUN_GUIDE shell steps for full KG construction.

In [1]:
from pathlib import Path
import sys
import importlib.util

# Make sure the local PyHealth package is importable from this notebook.
# Notebook location: PyHealth/examples/rexkg/load_dataset.ipynb
# Package root:      PyHealth/
pyhealth_root = Path.cwd().resolve().parents[1]
if str(pyhealth_root) not in sys.path:
    sys.path.insert(0, str(pyhealth_root))

from pyhealth.datasets import RexKGDataset #, split_by_patient, get_dataloader
from pyhealth.tasks import (
    RexKGEntityExtractionRadiology,
    RexKGRelationExtractionRadiology,
    RexKGKnowledgeGraphConstruction,
)
from pyhealth.models import RexKG

/home/strawhat/miniconda3/envs/cs598-pyhealth/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1) Configure Paths

Set `PROJECT_ROOT` to your repo root if auto-detection does not match your environment.

In [2]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "PyHealth").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

REXKG_DATA_ROOT = PROJECT_ROOT / "src" / "ner" / "data"
print("PROJECT_ROOT:", PROJECT_ROOT)
print("REXKG_DATA_ROOT:", REXKG_DATA_ROOT)
print("Data root exists:", REXKG_DATA_ROOT.exists())

PROJECT_ROOT: /mnt/c/Users/Aarje/OneDrive - University of Illinois - Urbana/School/cs598_project/PyHealth/examples
REXKG_DATA_ROOT: /mnt/c/Users/Aarje/OneDrive - University of Illinois - Urbana/School/cs598_project/PyHealth/examples/src/ner/data
Data root exists: False


## 2) Load RexKGDataset - Data Preperation

In [3]:
# data_dir = PROJECT_ROOT / "src" / "ner" / "data"
print("Using PROJECT_ROOT:", PROJECT_ROOT)

expected = PROJECT_ROOT / "df_chexpert_plus_240401.csv"
if not expected.exists():
    raise FileNotFoundError(f"Missing expected CSV: {expected}")

dataset = RexKGDataset(root=str(expected))
dataset.stats()

Using PROJECT_ROOT: /mnt/c/Users/Aarje/OneDrive - University of Illinois - Urbana/School/cs598_project/PyHealth/examples
No config path provided, using default RexKG config
Initializing rexkg dataset from /mnt/c/Users/Aarje/OneDrive - University of Illinois - Urbana/School/cs598_project/PyHealth/examples (dev mode: False)
No cache_dir provided. Using default cache dir: /home/strawhat/.cache/pyhealth/77e4c830-4a0e-5eaf-a252-9cdbff96ceeb
Found cached event dataframe: /home/strawhat/.cache/pyhealth/77e4c830-4a0e-5eaf-a252-9cdbff96ceeb/global_event_df.parquet
Dataset: rexkg
Dev mode: False
Number of patients: 27361
Number of events: 59441


## 3) Apply ReXKG Tasks - Node and Edge Construction

In [ ]:
entity_task = RexKGEntityExtractionRadiology()
relation_task = RexKGRelationExtractionRadiology()
kg_task = RexKGKnowledgeGraphConstruction()

# Prefer repo-relative split files created by src/ner/data/structure_data.py
split_root = PROJECT_ROOT / "src" / "ner" / "data" / "data_split"
if not split_root.exists():
    # Fallback when running from PyHealth/examples/rexkg
    split_root = Path.cwd() / "data" / "data_split"
train_json = split_root / "train.json"
dev_json = split_root / "test.json"
test_json = split_root / "test.json"

for p in [train_json, dev_json, test_json]:
    if not p.exists():
        raise FileNotFoundError(f"Missing split file: {p}")

# Force output under this notebook folder.
entity_output_dir = Path.cwd() / "result" / "run_entity"
metrics = RexKGEntityExtractionRadiology.run_entity_pipeline(
    train_data=str(train_json),
    dev_data=str(dev_json),
    test_data=str(test_json),
    model="bert-base-uncased",
    output_dir=str(entity_output_dir),
    do_train=True,
    do_eval=True,
    eval_test=True,
    learning_rate=1e-5,
    task_learning_rate=5e-4,
    train_batch_size=8,
    eval_batch_size=64,
    num_epoch=1,
    context_window=100,
)
print(metrics)

pred_file = entity_output_dir / "ent_pred_mimic_headct.json"
print("Expected prediction file:", pred_file)
print("Prediction file exists:", pred_file.exists())

entity_samples = dataset.set_task(entity_task)
# relation_samples = dataset.set_task(relation_task)
# kg_samples = dataset.set_task(kg_task)

# print("Entity samples:", len(entity_samples))
# print("Relation samples:", len(relation_samples))
# print("KG samples:", len(kg_samples))

Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/mnt/c/Users/Aarje/OneDrive - University of Illinois - Urbana/School/cs598_project/PyHealth/pyhealth/tasks/rexkg.py:258: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  seed=seed,


Epoch,Training Loss,Validation Loss
1,0.351400,0.335372


{'output_dir': '/mnt/c/Users/Aarje/OneDrive - University of Illinois - Urbana/School/cs598_project/PyHealth/examples/rexkg/result/run_entity', 'train_metrics': {'train_runtime': 68.9898, 'train_samples_per_second': 73.721, 'train_steps_per_second': 9.219, 'total_flos': 259596783166800.0, 'train_loss': 0.5446638596132867, 'epoch': 1.0}, 'dev_metrics': {'eval_loss': 0.3353719115257263, 'eval_runtime': 1.776, 'eval_samples_per_second': 315.32, 'eval_steps_per_second': 5.068, 'epoch': 1.0}, 'test_metrics': {'test_loss': 0.3353719115257263, 'test_runtime': 1.4451, 'test_samples_per_second': 387.504, 'test_steps_per_second': 6.228, 'epoch': 1.0}}
Setting task rexkg_entity_extraction_radiology for rexkg base dataset...
Task cache paths: task_df=/home/strawhat/.cache/pyhealth/77e4c830-4a0e-5eaf-a252-9cdbff96ceeb/tasks/rexkg_entity_extraction_radiology_2f4037a3-7f62-5908-ae68-ceaebc1d8de6/task_df.ld, samples=/home/strawhat/.cache/pyhealth/77e4c830-4a0e-5eaf-a252-9cdbff96ceeb/tasks/rexkg_entity_

In [ ]:
if len(entity_samples) > 0:
    first = entity_samples[0]
    print("First entity sample keys:", sorted(first.keys()))
    print("Text chars:", len(first.get("text", "")))
else:
    print("No entity samples found. Check source CSV and text columns.")

## 4) Split + DataLoaders (PyHealth style)

Using entity task samples for downstream model/data checks.

In [ ]:
train_ds, val_ds, test_ds = split_by_patient(entity_samples, [0.7, 0.1, 0.2])

train_loader = get_dataloader(train_ds, batch_size=4, shuffle=True)
val_loader = get_dataloader(val_ds, batch_size=4, shuffle=False)
test_loader = get_dataloader(test_ds, batch_size=4, shuffle=False)

print("train/val/test sizes:", len(train_ds), len(val_ds), len(test_ds))

## 5) ReXKG Model Forward Smoke Test

This checks model wiring with a batch from the entity dataloader.

In [ ]:
import torch

model = RexKG(dataset=entity_samples, freeze_encoder=True)
batch = next(iter(train_loader))

if "text" not in batch:
    raise KeyError("Batch does not contain 'text'. Check task/input schema.")

with torch.no_grad():
    out = model(text=batch["text"])

print("Output keys:", sorted(out.keys()))
print("Entity logits shape:", tuple(out["entity_logits"].shape))
print("Relation logits shape:", tuple(out["relation_logits"].shape))

## 6) Optional: RUN_GUIDE Shell Pipeline

These commands are the original script-based ReXKG flow from `RUN_GUIDE.md`.
Run them from a terminal when you want full KG artifacts.

In [ ]:
run_guide_commands = [
    "cd src/ner/data && python structure_data.py",
    "cd src/ner && sh run_entity.sh",
    "cd src/ner && sh run_relation.sh",
    "cd src/ner && sh run_inference.sh",
    "cd src/ner/result/run_relation && python reverse_structure_data.py --input_json_file ../run_relation/predictions.json --save_json_file ../../data/your_test_file.json",
    "cd src/kg_construct/code && python get_entities.py --ent_pred_mimic_headct ../../ner/data/your_test_file.json --ent_real_pred_mimic_headct ../../ner/data/your_test_file.json --save_entity_dir ../result/your_run/entities --save_real_dir ../result/your_run/relation",
    "cd src/kg_construct/code && python get_umls_entities.py --save_entity_dir ../result/your_run/entities",
    "cd src/kg_construct/code && python filter_cui.py --save_entity_dir ../result/your_run/entities",
    "cd src/kg_construct/code && python structure_entities.py --save_entity_dir ../result/your_run/entities --ignore_count 10",
    "cd src/kg_construct/code && python get_kg_nodes.py --save_entity_dir ../result/your_run/entities --save_real_dir ../result/your_run/relation --save_kg_dir ../result/your_run/kg",
    "cd src/kg_construct/code && python get_size_relations.py --entity_dir ../result/your_run/entities --real_dir ../result/your_run/relation",
]

for c in run_guide_commands:
    print(c)